# Airbnb en Londres - preparación, limpieza y análisis descriptivo inicial -

Este notebook construye la base analítica principal del proyecto a partir del archivo de listings de Airbnb en Londres.  

Se realizan tareas de lectura, depuración, filtrado y análisis descriptivo para generar una primera caracterización del mercado según `price`, `room_type`, `minimum_nights`, `availability_365` y `neighbourhood`.

## Librerías
Se importan las librerías principales para trabajar con rutas, tablas, cálculos numéricos y gráficos.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Plotly (shareable / exportable)
import plotly.express as px
import plotly.graph_objects as go

import warnings

warnings.filterwarnings("ignore")

## Directorio del proyecto
Se define la carpeta raíz del proyecto para que el notebook funcione tanto desde `notebooks` como desde la carpeta principal.

In [2]:
from pathlib import Path

cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

print("Project dir:", PROJECT_DIR.name)
print("Working dir:", cwd.name)

Project dir: 2026-02_airbnb-london
Working dir: notebooks


## Rutas de trabajo
Se establecen las rutas de las carpetas de datos y de los archivos que se van a usar en el análisis.

In [3]:
DATA_STAGING = PROJECT_DIR / "data" / "staging"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"

LISTINGS_FILE = DATA_STAGING / "london-airbnb-listings-2025_09_14.xlsx"

## Lectura del archivo
Se carga el archivo de listings y se muestra una vista inicial para revisar tamaño y primeras filas.

In [4]:
df_listings = pd.read_excel(LISTINGS_FILE)
print("listings shape:", df_listings.shape)
df_listings.head()

listings shape: (96870, 18)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,13913,Holiday London DB Room Let-on going,54730,Alina,NaN,Islington,51.56861,-0.11270,Private room,70.0,1,55,2025-08-21,0.30,2,331,10,NaN
1,15400,Bright Chelsea Apartment. Chelsea!,60302,Philippa,NaN,Kensington and Chelsea,51.48780,-0.16813,Entire home/apt,149.0,4,97,2025-04-05,0.51,1,199,1,NaN
2,17402,Very Central Modern 3-Bed/2 Bath By Oxford St W1,67564,Liz,NaN,Westminster,51.52195,-0.14094,Entire home/apt,411.0,3,56,2024-02-19,0.32,2,80,0,NaN
3,24328,Battersea live/work artist house,41759,Joe,NaN,Wandsworth,51.47072,-0.16266,Entire home/apt,NaN,7,95,2025-07-05,0.53,1,294,1,NaN
4,36274,Bright 1 bedroom apt off brick lane in Shoreditch,133271,Hendryks,NaN,Tower Hamlets,51.52322,-0.06979,Entire home/apt,210.0,5,15,2025-09-06,0.09,2,323,6,NaN


## Instalación opcional
Celda comentada para instalar una dependencia solo si el entorno local la necesitara.

In [5]:
#!pip install pyod

## Limpieza de columnas
Se eliminan columnas que no se van a usar para dejar el dataset más limpio y manejable.

In [6]:
cols_drop = [
    "name",
    "host_id",
    "host_name",
    "neighbourhood_group",
    "license",
]
df_listings.drop(columns=cols_drop, inplace=True, errors="ignore")

## Revisión de columnas
Se listan las columnas disponibles después de la limpieza inicial.

In [7]:
df_listings.columns.tolist()

['id',
 'neighbourhood',
 'latitude',
 'longitude',
 'room_type',
 'price',
 'minimum_nights',
 'number_of_reviews',
 'last_review',
 'reviews_per_month',
 'calculated_host_listings_count',
 'availability_365',
 'number_of_reviews_ltm']

## Medianas rápidas
Se calculan las medianas de las variables numéricas principales para tener una referencia general del dataset.

In [8]:
cols = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
    "number_of_reviews_ltm"
]

df_listings[cols].median()

price                             135.00
minimum_nights                      2.00
number_of_reviews                   5.00
reviews_per_month                   0.52
calculated_host_listings_count      2.00
availability_365                   96.00
number_of_reviews_ltm               0.00
dtype: float64

## Precio por tipo de alojamiento
Se grafica un boxplot de precios por `room_type` y se muestra una tabla resumen con cantidad, mínimo, cuartiles, mediana y máximo.

In [9]:
df_plot = df_listings.copy()
df_plot = df_plot[df_plot["price"].notna() & (df_plot["price"] > 0)]
df_plot = df_plot[df_plot["room_type"].notna()]

preferred_order = ["Entire home/apt", "Private room", "Hotel room", "Shared room"]
present = [x for x in preferred_order if x in df_plot["room_type"].unique()]
others = sorted([x for x in df_plot["room_type"].unique() if x not in present])
order = present + others

fig = px.box(
    df_plot,
    y="room_type",
    x="price",
    color="room_type",
    category_orders={"room_type": order},
    points=False,
    title="Distribución de precios por tipo de alojamiento"
)

fig.update_xaxes(range=[0, df_plot["price"].quantile(0.99)])
fig.update_layout(
    xaxis_title="Precio por noche (GBP)",
    yaxis_title="Tipo de alojamiento",
    legend_title="Tipo de alojamiento",
    height=600
)

fig.show()

tabla_resumen = (
    df_plot.groupby("room_type")["price"]
    .agg(
        cantidad="count",
        minimo="min",
        q1=lambda x: x.quantile(0.25),
        mediana="median",
        q3=lambda x: x.quantile(0.75),
        maximo="max"
    )
    .reindex(order)
    .round(0)
)

tabla_resumen

,cantidad,minimo,q1,mediana,q3,maximo
room_type,,,,,,
Entire home/apt,42318,9.0,122.0,175.0,266.0,1085147.0
Private room,19382,8.0,46.0,61.0,85.0,66189.0
Hotel room,72,19.0,112.0,281.0,925.0,7377.0
Shared room,191,7.0,26.0,32.0,56.0,1800.0


## Creación de la base analítica
Se convierte `last_review` a fecha y se construye `df_london_core` aplicando los filtros principales del análisis.

In [10]:
df_listings["last_review"] = pd.to_datetime(
    df_listings["last_review"],
    format="%d/%m/%Y",
    errors="coerce"
)

cutoff_last_review = pd.Timestamp("2023-09-14")

df_london_core = df_listings[
    (df_listings["price"] > 0) &
    (df_listings["last_review"] > cutoff_last_review) &
    (df_listings["minimum_nights"] < 90) &
    (df_listings["availability_365"] > 0)
].copy()

print("df_listings original:", df_listings.shape)
print("df_london_core:", df_london_core.shape)
print("Porcentaje que queda:", round(len(df_london_core) / len(df_listings) * 100, 2), "%")

df_listings original: (96870, 13)
df_london_core: (44019, 13)
Porcentaje que queda: 45.44 %


## Indicadores principales
Se calculan las medianas de las variables clave y se muestran como KPIs visuales para resumir la base filtrada.

In [11]:
import math
from plotly.subplots import make_subplots
import plotly.graph_objects as go

kpis = [
    {"col": "price",                          "title": "Precio mediano",                "suffix": " £",      "format": ",.0f"},
    {"col": "minimum_nights",                 "title": "Estancia mínima (mediana)",     "suffix": " noches", "format": ",.0f"},
    {"col": "number_of_reviews",              "title": "Reseñas totales (mediana)",     "suffix": "",        "format": ",.0f"},
    {"col": "reviews_per_month",              "title": "Reseñas por mes (mediana)",     "suffix": " /mes",   "format": ",.2f"},
    {"col": "calculated_host_listings_count", "title": "Listings por host (mediana)",   "suffix": "",        "format": ",.0f"},
    {"col": "availability_365",               "title": "Disponibilidad anual (mediana)","suffix": " días",   "format": ",.0f"},
    {"col": "number_of_reviews_ltm",          "title": "Reseñas últimos 12 meses",      "suffix": "",        "format": ",.0f"},
]

for k in kpis:
    k["value"] = df_london_core[k["col"]].median()

n = len(kpis)
rows = math.ceil(n / 2)

fig = make_subplots(
    rows=rows,
    cols=2,
    specs=[[{"type": "indicator"}, {"type": "indicator"}] for _ in range(rows)],
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

for i, k in enumerate(kpis):
    row = (i // 2) + 1
    col = (i % 2) + 1

    fig.add_trace(
        go.Indicator(
            mode="number",
            value=k["value"],
            title={"text": k["title"], "font": {"size": 16}},
            number={
                "valueformat": k["format"],
                "suffix": k["suffix"],
                "font": {"size": 50}
            },
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title={"text": "KPIs de medianas — Airbnb Londres", "x": 0.5},
    height=rows * 170,
    margin=dict(l=30, r=30, t=70, b=30),
)

fig.show()

## Boxplot sobre la base final
Se repite el análisis de precios por tipo de alojamiento, pero ya usando `df_london_core` como base definitiva.

In [12]:
df_plot = df_london_core.copy()

preferred_order = ["Entire home/apt", "Private room", "Hotel room", "Shared room"]
present = [x for x in preferred_order if x in df_plot["room_type"].unique()]
others = sorted([x for x in df_plot["room_type"].unique() if x not in present])
order = present + others

fig = px.box(
    df_plot,
    y="room_type",
    x="price",
    color="room_type",
    category_orders={"room_type": order},
    points=False,
    title="Distribución de precios por tipo de alojamiento"
)

fig.update_xaxes(range=[0, df_plot["price"].quantile(0.99)])

fig.update_layout(
    xaxis_title="Precio por noche (GBP)",
    yaxis_title="Tipo de alojamiento",
    legend_title="Tipo de alojamiento",
    height=600
)

fig.show()

tabla_resumen = (
    df_plot.groupby("room_type")["price"]
    .agg(
        cantidad="count",
        minimo="min",
        q1=lambda x: x.quantile(0.25),
        mediana="median",
        q3=lambda x: x.quantile(0.75),
        maximo="max"
    )
    .reindex(order)
    .round(0)
)

tabla_resumen

,cantidad,minimo,q1,mediana,q3,maximo
room_type,,,,,,
Entire home/apt,30595,11.0,119.0,167.0,246.0,15000.0
Private room,13242,10.0,45.0,59.0,79.0,66189.0
Hotel room,37,24.0,101.0,198.0,437.0,1490.0
Shared room,145,13.0,26.0,30.0,46.0,1800.0


## Histogramas de precio por tipo
Se comparan las distribuciones de precio por `room_type` con histogramas, línea suavizada y agrupación de valores altos en el tope definido.

In [13]:
preferred_order = ["Entire home/apt", "Private room", "Hotel room", "Shared room"]
present = [x for x in preferred_order if x in df_london_core["room_type"].unique()]
others = sorted([x for x in df_london_core["room_type"].unique() if x not in present])
room_order = present + others

palette = px.colors.qualitative.Plotly
color_map = {rt: palette[i % len(palette)] for i, rt in enumerate(room_order)}

def hex_to_rgba(hex_color, alpha=0.25):
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

kernel = np.array([0.25, 0.50, 0.25])

def smooth_weighted_3(y):
    y_pad = np.pad(y, (1, 1), mode="edge")
    return np.convolve(y_pad, kernel, mode="valid")

p99_by_rt = {
    rt: df_london_core.loc[df_london_core["room_type"] == rt, "price"].quantile(0.99)
    for rt in room_order
}

PRICE_CAP = 800

NBINS = 200
bins = np.linspace(0, PRICE_CAP, NBINS + 1)
centers = (bins[:-1] + bins[1:]) / 2
bin_width = bins[1] - bins[0]

hist_by_rt = {}
smooth_by_rt = {}
ymax = 0

for rt in room_order:
    x = df_london_core.loc[df_london_core["room_type"] == rt, "price"]

    # recorte visual por P99 de cada room_type
    x = x[x <= p99_by_rt[rt]]

    # todos los valores mayores a 800 se agrupan en 800
    x = x.clip(upper=PRICE_CAP)

    hist, _ = np.histogram(x, bins=bins, density=True)
    smooth = smooth_weighted_3(hist)

    hist_by_rt[rt] = hist
    smooth_by_rt[rt] = smooth
    ymax = max(ymax, float(np.nanmax(hist)), float(np.nanmax(smooth)))

rows = len(room_order)

fig = make_subplots(
    rows=rows,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=room_order
)

for i, rt in enumerate(room_order, start=1):
    c = color_map[rt]

    fig.add_trace(
        go.Bar(
            x=centers,
            y=hist_by_rt[rt],
            marker=dict(color=hex_to_rgba(c, 0.20), line=dict(color=c, width=1)),
            width=bin_width * 0.95,
            showlegend=False
        ),
        row=i, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=centers,
            y=smooth_by_rt[rt],
            mode="lines",
            line=dict(width=4, color=c),
            showlegend=False
        ),
        row=i, col=1
    )

fig.update_xaxes(range=[0, PRICE_CAP])
fig.update_yaxes(range=[0, ymax * 1.05])

fig.update_layout(
    title="Distribución de precios por tipo de alojamiento — histograma + línea suavizada (valores > 800 agrupados en 800)",
    height=280 * rows + 120,
    margin=dict(l=70, r=30, t=90, b=60),
    bargap=0.05
)

fig.update_xaxes(title_text="Precio por noche (GBP)", row=rows, col=1)
fig.update_yaxes(title_text="Densidad", row=1, col=1)

fig.show()

## Distribución en escala logarítmica
Se transforma el precio con `log(1 + price)` para comparar mejor la forma de las distribuciones entre tipos de alojamiento.

In [14]:
# Solo quitamos nulos de room_type
df_plot = df_london_core[df_london_core["room_type"].notna()].copy()

# Transformación log(1 + price)
df_plot["price_log"] = np.log1p(df_plot["price"])

preferred_order = ["Entire home/apt", "Private room", "Hotel room", "Shared room"]
present = [x for x in preferred_order if x in df_plot["room_type"].unique()]
others = sorted([x for x in df_plot["room_type"].unique() if x not in present])
room_order = present + others

palette = px.colors.qualitative.Plotly
color_map = {rt: palette[i % len(palette)] for i, rt in enumerate(room_order)}

def hex_to_rgba(hex_color, alpha=0.25):
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

kernel = np.array([0.25, 0.50, 0.25])

def smooth_weighted_3(y):
    y_pad = np.pad(y, (1, 1), mode="edge")
    return np.convolve(y_pad, kernel, mode="valid")

NBINS = 200
xmin = df_plot["price_log"].min()
xmax = df_plot["price_log"].max()

bins = np.linspace(xmin, xmax, NBINS + 1)
centers = (bins[:-1] + bins[1:]) / 2
bin_width = bins[1] - bins[0]

hist_by_rt = {}
smooth_by_rt = {}
ymax = 0

for rt in room_order:
    x = df_plot.loc[df_plot["room_type"] == rt, "price_log"]

    hist, _ = np.histogram(x, bins=bins, density=True)
    smooth = smooth_weighted_3(hist)

    hist_by_rt[rt] = hist
    smooth_by_rt[rt] = smooth
    ymax = max(ymax, float(np.nanmax(hist)), float(np.nanmax(smooth)))

rows = len(room_order)

fig = make_subplots(
    rows=rows,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=room_order
)

for i, rt in enumerate(room_order, start=1):
    c = color_map[rt]

    fig.add_trace(
        go.Bar(
            x=centers,
            y=hist_by_rt[rt],
            marker=dict(color=hex_to_rgba(c, 0.20), line=dict(color=c, width=1)),
            width=bin_width * 0.95,
            showlegend=False
        ),
        row=i, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=centers,
            y=smooth_by_rt[rt],
            mode="lines",
            line=dict(width=4, color=c),
            showlegend=False
        ),
        row=i, col=1
    )

fig.update_yaxes(range=[0, ymax * 1.05])

fig.update_layout(
    title="Distribución de precios en escala logarítmica por tipo de alojamiento",
    height=280 * rows + 120,
    margin=dict(l=70, r=30, t=90, b=60),
    bargap=0.05
)

fig.update_xaxes(title_text="log(1 + price)", row=rows, col=1)
fig.update_yaxes(title_text="Densidad", row=1, col=1)

fig.show()

## Barrios con mayor precio mediano
Se calcula el precio mediano por barrio y se grafica el ranking de los barrios más caros.

In [15]:
TOP_N = 25

medianas_barrio = (
    df_london_core[["neighbourhood", "price"]]
    .dropna()
    .groupby("neighbourhood")["price"]
    .median()
    .sort_values(ascending=False)
    .head(TOP_N)
)

n = len(medianas_barrio)
base_colors = px.colors.sample_colorscale(
    "Turbo",
    [i / (n - 1) if n > 1 else 0.5 for i in range(n)]
)

def rgb_to_rgba(rgb_str, alpha):
    return rgb_str.replace("rgb(", "rgba(").replace(")", f",{alpha})")

fill_colors = [rgb_to_rgba(c, 0.25) for c in base_colors]
line_colors = [rgb_to_rgba(c, 1.0) for c in base_colors]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=medianas_barrio.values,
        y=medianas_barrio.index,
        orientation="h",
        marker=dict(color=fill_colors, line=dict(color=line_colors, width=2)),
        text=[f"{v:,.0f} £" for v in medianas_barrio.values],
        textposition="outside",
        cliponaxis=False,
        showlegend=False
    )
)

fig.update_layout(
    title=f"Precio mediano por barrio (Top {TOP_N})",
    xaxis_title="Precio por noche (GBP)",
    yaxis_title="Barrio",
    bargap=0.35,
    height=900,
    margin=dict(l=180, r=80, t=80, b=60)
)

fig.update_yaxes(autorange="reversed")

fig.show()

## Mapa de calor barrio por tipo
Se arma un heatmap con el precio mediano por barrio y tipo de alojamiento, normalizando el color dentro de cada columna.

In [16]:
TOP_N = 40

df_heatmap = df_london_core[["neighbourhood", "room_type", "price"]].dropna()

top_neigh = df_heatmap["neighbourhood"].value_counts().head(TOP_N).index
df_heatmap = df_heatmap[df_heatmap["neighbourhood"].isin(top_neigh)]

pivot = pd.pivot_table(
    df_heatmap,
    index="neighbourhood",
    columns="room_type",
    values="price",
    aggfunc="median"
)

col_max = pivot.max(axis=0)
pivot_norm = pivot.divide(col_max, axis=1)

text = pivot.copy()
for col in text.columns:
    text[col] = text[col].apply(lambda v: "" if pd.isna(v) else f"£{v:,.0f}")

fig = go.Figure(
    data=go.Heatmap(
        z=pivot_norm.values,
        x=pivot_norm.columns.astype(str),
        y=pivot_norm.index.astype(str),
        colorscale="Viridis",
        zmin=0,
        zmax=1,
        text=text.values,
        texttemplate="%{text}",
        hovertemplate=(
            "Barrio: %{y}<br>"
            "Tipo de alojamiento: %{x}<br>"
            "Precio mediano: %{customdata}<br>"
            "Escala normalizada: %{z:.2f}<extra></extra>"
        ),
        customdata=text.values
    )
)

fig.update_layout(
    title="Mapa de calor — precio mediano por barrio y tipo de alojamiento",
    xaxis_title="Tipo de alojamiento",
    yaxis_title="Barrio",
    height=900,
    margin=dict(l=180, r=40, t=80, b=60)
)

fig.show()

## Precio según estancia mínima
Se agrupan los listings por bandas de `minimum_nights` y se compara el precio mediano por tipo de alojamiento.

In [17]:
import colorsys

bands = ["1–2", "3–6", "7–89"]

band_series = pd.cut(
    df_london_core["minimum_nights"],
    bins=[0, 2, 6, 89],
    labels=bands,
    include_lowest=True
)

df_bandas = pd.DataFrame({
    "room_type": df_london_core["room_type"],
    "band": band_series,
    "price": df_london_core["price"]
})

pivot = (
    df_bandas.groupby(["room_type", "band"])["price"]
    .median()
    .unstack()
)

preferred_order = ["Entire home/apt", "Private room", "Hotel room", "Shared room"]
present = [x for x in preferred_order if x in pivot.index]
others = sorted([x for x in pivot.index if x not in present])
room_order = present + others

pivot = pivot.reindex(index=room_order, columns=bands)

palette = px.colors.qualitative.Plotly
base_color = {rt: palette[i % len(palette)] for i, rt in enumerate(room_order)}

def shade_hex(hex_color, t):
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16) / 255
    g = int(h[2:4], 16) / 255
    b = int(h[4:6], 16) / 255

    hh, ll, ss = colorsys.rgb_to_hls(r, g, b)
    ll_new = 0.80 - 0.45 * t
    ll_new = max(0.20, min(0.90, ll_new))

    r2, g2, b2 = colorsys.hls_to_rgb(hh, ll_new, ss)
    return "#{:02x}{:02x}{:02x}".format(int(r2 * 255), int(g2 * 255), int(b2 * 255))

intensity = pd.DataFrame(index=pivot.index, columns=pivot.columns, dtype=float)

for rt in pivot.index:
    vals = pivot.loc[rt].dropna()
    if len(vals) <= 1:
        intensity.loc[rt] = 0.5
    else:
        vmin, vmax = vals.min(), vals.max()
        intensity.loc[rt] = 0.5 if vmax == vmin else (pivot.loc[rt] - vmin) / (vmax - vmin)

fig = go.Figure()

for band in bands:
    s = pivot[band].dropna()
    if s.empty:
        continue

    y = s.index.tolist()
    x = s.values

    colors_fill = []
    colors_line = []
    texts = []

    for rt, v in zip(y, x):
        t = float(intensity.loc[rt, band]) if pd.notna(intensity.loc[rt, band]) else 0.5
        c_fill = shade_hex(base_color[rt], t)
        c_line = shade_hex(base_color[rt], min(1, t + 0.25))
        colors_fill.append(c_fill)
        colors_line.append(c_line)
        texts.append(f"{v:,.0f} £ — {band} noches")

    fig.add_trace(
        go.Bar(
            name=f"{band} noches",
            y=y,
            x=x,
            orientation="h",
            marker=dict(color=colors_fill, line=dict(color=colors_line, width=2)),
            text=texts,
            textposition="outside",
            cliponaxis=False
        )
    )

fig.update_layout(
    title="Precio mediano por tipo de alojamiento y tramo de estancia mínima",
    xaxis_title="Precio mediano por noche (GBP)",
    yaxis_title="Tipo de alojamiento",
    barmode="group",
    bargap=0.30,
    bargroupgap=0.10,
    height=650,
    margin=dict(l=120, r=140, t=80, b=60),
)

fig.update_yaxes(categoryorder="array", categoryarray=room_order, autorange="reversed")

fig.show()

## Precio según disponibilidad anual
Se agrupa `availability_365` en tramos y se compara el precio mediano por tipo de alojamiento.

In [18]:
df_plot = df_london_core.copy()

avail_order = ["1–30", "31–90", "91–180", "181–365"]

df_plot["avail_bin"] = pd.cut(
    df_plot["availability_365"],
    bins=[0, 30, 90, 180, 365],
    labels=avail_order,
    include_lowest=True
)

agg = (
    df_plot.dropna(subset=["avail_bin", "room_type", "price"])
    .groupby(["room_type", "avail_bin"])["price"]
    .median()
    .reset_index()
)

preferred_order = ["Entire home/apt", "Private room", "Hotel room", "Shared room"]
present = [x for x in preferred_order if x in agg["room_type"].unique()]
others = sorted([x for x in agg["room_type"].unique() if x not in present])
room_order = present + others

fig = px.bar(
    agg,
    x="price",
    y="room_type",
    color="avail_bin",
    barmode="group",
    orientation="h",
    category_orders={
        "room_type": room_order,
        "avail_bin": avail_order
    },
    title="Precio mediano por tipo de alojamiento y tramo de disponibilidad anual"
)

fig.update_layout(
    xaxis_title="Precio mediano por noche (GBP)",
    yaxis_title="Tipo de alojamiento",
    legend_title="Disponibilidad (días)"
)

fig.show()

## Tabla resumen por barrio
Se construye una tabla agregada por `neighbourhood` con cantidad de listings y percentiles de precio.

In [19]:
df_neigh = (
    df_london_core[["neighbourhood", "price"]]
    .dropna()
    .groupby("neighbourhood")["price"]
    .agg(
        n_listings="size",
        price_p01=lambda s: s.quantile(0.01),
        price_p05=lambda s: s.quantile(0.05),
        price_p10=lambda s: s.quantile(0.10),
        price_p25=lambda s: s.quantile(0.25),
        price_median="median",
        price_p75=lambda s: s.quantile(0.75),
        price_p90=lambda s: s.quantile(0.90),
        price_p95=lambda s: s.quantile(0.95),
        price_p99=lambda s: s.quantile(0.99)
    )
    .reset_index()
)

df_neigh.head(10)

,neighbourhood,n_listings,price_p01,price_p05,price_p10,price_p25,price_median,price_p75,price_p90,price_p95,price_p99
0,Barking and Dagenham,344,25.00,30.0,32.3,41.00,78.5,144.25,192.1,224.85,334.55
1,Barnet,1161,31.00,38.0,45.0,65.00,102.0,151.00,213.0,277.00,403.80
2,Bexley,302,25.01,29.0,33.0,41.00,66.0,129.25,199.6,241.90,997.39
3,Brent,1441,30.00,37.0,42.0,59.00,102.0,165.00,245.0,330.00,500.00
4,Bromley,415,27.14,35.0,40.0,52.00,90.0,139.00,190.6,240.00,313.32
5,Camden,3270,40.00,54.0,65.0,96.00,155.0,236.00,341.0,433.00,841.24
6,City of London,294,58.00,95.0,125.6,159.25,215.0,286.00,358.0,428.90,544.07
7,Croydon,788,23.87,29.0,34.0,43.00,69.0,120.00,168.6,206.90,304.43
8,Ealing,1128,28.00,35.0,40.0,56.00,87.0,135.25,215.6,281.65,575.83
9,Enfield,438,27.00,32.0,35.0,46.00,77.0,129.00,199.3,243.20,303.78


## Columnas finales de trabajo
Se listan las columnas disponibles en `df_london_core` para verificar la estructura final de la base.

In [20]:
df_london_core.columns.tolist()

['id',
 'neighbourhood',
 'latitude',
 'longitude',
 'room_type',
 'price',
 'minimum_nights',
 'number_of_reviews',
 'last_review',
 'reviews_per_month',
 'calculated_host_listings_count',
 'availability_365',
 'number_of_reviews_ltm']

## Guardado opcional en Parquet
Celda comentada para exportar la base final en formato Parquet.

In [21]:
#   # Guardar en Parquet
#   out_parquet = DATA_PROCESSED / "df_london_core.parquet"
#   df_london_core.to_parquet(out_parquet, index=False)
#   
#   print("Guardado Parquet:", out_parquet)

## Guardado opcional en CSV
Celda comentada para exportar la base final en formato CSV.

In [22]:
# # Guardar en CSV
# out_csv = DATA_PROCESSED / "df_london_core.csv"
# df_london_core.to_csv(out_csv, index=False)
# 
# print("Guardado CSV:", out_csv)